# CAFE — Ablation Study (FAST)
### Incremental contribution analysis

| Method | Augmentation | Spatial Attention |
|---|---|---|
| BASELINE | basic | X |
| + Aug | **Albumentations** | X |
| + PerGroup_FA | basic | O (live landmarks, re-detected per batch) |
| + PerGroup_FA + Aug | **Albumentations** | O (live landmarks, re-detected per batch) |


## 1. Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install --upgrade numba
!pip install ftfy regex tqdm kagglehub==0.3.6 pandas scikit-learn matplotlib
!pip install git+https://github.com/openai/CLIP.git
!pip install face-alignment && pip uninstall opencv-python -y && pip install --force-reinstall --no-cache-dir opencv-python-headless
!pip install scipy
!pip install albumentations


## 2. Mount Google Drive
Upload `resnet18_msceleb.pth` to a `deeplearning` folder in Drive.  
**Skip this cell if using Elice.**

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')
#import os
#DRIVE_FOLDER = '/content/drive/MyDrive/deeplearning'

## 3. File Paths
Set paths to model weights and dataset.

In [ ]:
import os

if os.path.exists('/content/drive'):
  FILE_FOLDER = '/content/drive/MyDrive/deeplearning'
else:
  FILE_FOLDER = '.'

MSCELEB_PATH = os.path.join(FILE_FOLDER, 'resnet18_msceleb.pth')

!mkdir -p ~/.kaggle
!cp "{FILE_FOLDER}/kaggle.json" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

print('MS-Celeb weights found :', os.path.exists(MSCELEB_PATH))
print('Dataset downloads will be handled in the automation cell.')

In [ ]:
import os
import cv2
import math
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torchvision import transforms
from torch.autograd import Variable
from torch.nn.modules.module import Module
from torch.nn.modules.utils import _pair
from torch.nn.parameter import Parameter
from torch.utils.data import random_split
from collections import Counter

import face_alignment
from scipy.ndimage import gaussian_filter
import clip
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

clip_model, preprocess = clip.load('ViT-B/32', device=device)
print('CLIP loaded.')

fa = face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D, device=str(device))
print('face_alignment loaded.')


In [ ]:
raf_map = {
    '1': 5,  # surprise
    '2': 2,  # fear
    '3': 1,  # disgust
    '4': 3,  # happy
    '5': 4,  # sad
    '6': 0,  # angry
    '7': 6   # neutral
}

ferplus_map = {
    'fear': 2,
    'suprise': 5,
    'surprise': 5,  # test folder correct spelling
    'angry': 0,
    'neutral': 6,
    'sad': 4,
    'disgust': 1,
    'happy': 3
}

affectnet_train_map = {
    'anger': 0,
    'disgust': 1,
    'fear': 2,
    'happy': 3,
    'neutral': 6,
    'sad': 4,
    'surprise': 5
}

affectnet_test_map = {
    'Anger': 0,
    'disgust': 1,
    'fear': 2,
    'happy': 3,
    'neutral': 6,
    'sad': 4,
    'surprise': 5
}

sfew_map = {
    'Angry': 0,
    'Disgust': 1,
    'Fear': 2,
    'Happy': 3,
    'Neutral': 6,
    'Sad': 4,
    'Surprise': 5
}

mma_map = {
    'angry': 0,
    'disgust': 1,
    'fear': 2,
    'happy': 3,
    'neutral': 6,
    'sad': 4,
    'surprise': 5
}

## 5a. Augmentation & Transforms

In [ ]:
def add_g(image_array, mean=0.0, var=30):
    std = var ** 0.5
    image_add = image_array + np.random.normal(mean, std, image_array.shape)
    image_add = np.clip(image_add, 0, 255).astype(np.uint8)
    return image_add

def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True


class FolderDataset(data.Dataset):

    def __init__(self, root, phase='train', transform=None, label_map=None, use_albu=False):
        self.transform = transform
        self.phase     = phase
        self.label_map = label_map
        self.use_albu  = use_albu  # True if transform is Albumentations

        split_dir = os.path.join(root, phase)
        self.classes = sorted([c for c in os.listdir(split_dir) if (c != 'contempt' and c != 'Contempt')])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        self.file_paths = []
        self.labels     = []

        for cls in self.classes:
            cls_dir = os.path.join(split_dir, cls)
            if not os.path.isdir(cls_dir): continue
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.file_paths.append(os.path.join(cls_dir, fname))
                    if self.label_map is not None:
                        mapped_label = self.label_map[cls]
                    else:
                        mapped_label = self.class_to_idx[cls]
                    self.labels.append(mapped_label)

        print(f'[{phase}] {len(self.file_paths)} samples | classes: {self.classes}')

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        label = self.labels[idx]
        image = cv2.imread(self.file_paths[idx])
        image = image[:, :, ::-1].copy()  # BGR -> RGB

        if self.transform is not None:
            if self.use_albu:
                image = self.transform(image=image)['image']
            else:
                image = self.transform(image)

        image_flip = torch.flip(image, dims=[2])
        return image, label, idx, image_flip


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Basic augmentation (BASELINE + PerGroup)
basic_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomErasing(scale=(0.02, 0.25))
])

# Strong augmentation (Albumentations)
albu_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.3),
    A.RandomResizedCrop((224, 224), scale=(0.9, 1.0), p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02, p=0.5),
    A.GaussNoise(std_range=(0.012, 0.022), p=0.2),  # albumentations>=2.0 API (std as fraction of 255; ~var 10-30)
    A.Resize(224, 224),
    A.Normalize(),
    ToTensorV2()
])

eval_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Keep backward compat
train_transforms = basic_transforms

print('Transforms defined: basic (torchvision), albu (Albumentations), eval.')


## 5b. AU-Guided Channel Relevance — 512-Channel Cosine Similarity

For each image: build a 7x7 AU heatmap from face_alignment landmarks + FACS codebook.  
Compare each of the 512 channel maps (7x7) against the heatmap via cosine similarity.  
High-scoring channels receive stronger separation pressure in l_sep.

In [ ]:
# ── AU Codebook + Landmark Heatmap Builder ───────────────────────────────────
# FACS-based mapping: expression -> active AUs -> landmark indices -> 7x7 heatmap

# Expression -> active AUs (FACS codebook)
AU_CODEBOOK = {
    'angry':    [4, 5, 7, 23, 24],
    'disgust':  [9, 15, 16, 25, 26],
    'fear':     [1, 2, 4, 5, 7, 20, 26],
    'happy':    [6, 12],
    'neutral':  [],
    'sad':      [1, 4, 15, 17],
    'surprise': [1, 2, 5, 26, 27],
    'suprise':  [1, 2, 5, 26, 27],  # FERPlus typo
}

# AU -> 68-point landmark indices (compatible with face_alignment)
AU_TO_DLIB = {
    1:  [17, 18, 19, 21, 22, 23, 24, 26],  # inner brow
    2:  [17, 18, 19, 20, 23, 24, 25, 26],  # outer brow
    4:  [19, 20, 21, 22, 23, 24],           # brow lowerer
    5:  [36, 37, 38, 43, 44, 45],           # upper lid (eyes)
    6:  [1, 2, 14, 15],                     # cheek raiser
    7:  [36, 39, 42, 45],                   # lid tightener
    9:  [27, 28, 29, 30],                   # nose wrinkler
    12: [48, 54, 49, 53],                   # lip corner puller (smile)
    15: [48, 54, 56, 58],                   # lip corner depressor
    16: [56, 57, 58],                       # lower lip depressor
    17: [6, 7, 8, 57],                      # chin raiser
    20: [48, 54],                           # lip stretcher
    23: [60, 61, 62, 63, 64],              # lip tightener
    24: [60, 61, 62, 63, 64],              # lip pressor
    25: [60, 61, 62, 63, 64, 65, 66, 67], # lips part
    26: [6, 7, 8, 56, 57, 58],            # jaw drop
    27: [6, 7, 8, 56, 57, 58],            # mouth stretch
}

SYNONYM_MAP = {'anger': 'angry', 'happiness': 'happy', 'sadness': 'sad', 'surprised': 'surprise'}

GRID_SIZE = 7
IMG_SIZE  = 224
SCALE     = GRID_SIZE / IMG_SIZE

# Unified label index -> expression name
# 0=Angry, 1=Disgust, 2=Fear, 3=Happy, 4=Sad, 5=Surprise, 6=Neutral
LABEL_TO_CLASS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

print('AU codebook defined.')
print(f'Grid scale: {IMG_SIZE}px image -> {GRID_SIZE}x{GRID_SIZE} feature map')

In [ ]:
# ── Landmark Detection + AU Heatmap Builder ──────────────────────────────────

_MEAN = np.array([0.485, 0.456, 0.406])
_STD  = np.array([0.229, 0.224, 0.225])

landmark_stats = {'total': 0, 'success': 0, 'fail': 0, 'neutral_skip': 0}

def reset_landmark_stats():
    landmark_stats['total'] = 0
    landmark_stats['success'] = 0
    landmark_stats['fail'] = 0
    landmark_stats['neutral_skip'] = 0

def denormalize_to_uint8(img_tensor):
    img_np = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img_np = (img_np * _STD + _MEAN) * 255.0
    return np.clip(img_np, 0, 255).astype(np.uint8)

# ── face_alignment landmark detection ────────────────────────────────
def get_landmarks_fa(img_np):
    try:
        landmarks_list = fa.get_landmarks(img_np)
    except Exception:
        return None
    if landmarks_list is None or len(landmarks_list) == 0:
        return None
    return landmarks_list[0]

# ── Shared: build (7,7) heatmap from landmarks + expression AUs ─────
def build_heatmap_from_aus(landmarks, expression_name, img_h, img_w):
    expr = SYNONYM_MAP.get(expression_name, expression_name)
    active_aus = AU_CODEBOOK.get(expr, [])
    if not active_aus:
        return torch.ones(GRID_SIZE, GRID_SIZE, dtype=torch.float32)
    heatmap = np.zeros((GRID_SIZE, GRID_SIZE), dtype=np.float32)
    for au in active_aus:
        for li in AU_TO_DLIB.get(au, []):
            if li < len(landmarks):
                lx, ly = landmarks[li]
                gc = int(np.clip(lx * GRID_SIZE / img_w, 0, GRID_SIZE - 1))
                gr = int(np.clip(ly * GRID_SIZE / img_h, 0, GRID_SIZE - 1))
                heatmap[gr, gc] += 1.0
    heatmap = gaussian_filter(heatmap, sigma=0.8)
    hmax = heatmap.max()
    if hmax > 0:
        heatmap = heatmap / hmax
    return torch.from_numpy(heatmap)

# ── Single heatmap (for Weighted_L_Sep, uses face_alignment) ────────
def build_heatmap_from_landmarks(img_np, expression_name):
    landmark_stats['total'] += 1
    expr = SYNONYM_MAP.get(expression_name, expression_name)
    if not AU_CODEBOOK.get(expr, []):
        landmark_stats['neutral_skip'] += 1
        return torch.ones(GRID_SIZE, GRID_SIZE, dtype=torch.float32)
    landmarks = get_landmarks_fa(img_np)
    if landmarks is None:
        landmark_stats['fail'] += 1
        return torch.ones(GRID_SIZE, GRID_SIZE, dtype=torch.float32)
    landmark_stats['success'] += 1
    return build_heatmap_from_aus(landmarks, expression_name, img_np.shape[0], img_np.shape[1])

# ── Per-Group Spatial Attention (512, 7, 7) ──────────────────────────
GROUP_SIZES = [73, 73, 73, 73, 73, 73, 74]
GROUP_EXPRS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

def _build_per_group(landmarks, img_h, img_w):
    """Shared logic: landmarks (68,2) -> (512, 7, 7) attention."""
    attention = torch.ones(512, GRID_SIZE, GRID_SIZE, dtype=torch.float32)
    ch_start = 0
    for expr, g_size in zip(GROUP_EXPRS, GROUP_SIZES):
        hm = build_heatmap_from_aus(landmarks, expr, img_h, img_w)
        attention[ch_start:ch_start+g_size] = hm
        ch_start += g_size
    return attention

def build_per_group_attention_fa(img_np):
    """Per-group attention using face_alignment."""
    landmark_stats['total'] += 1
    landmarks = get_landmarks_fa(img_np)
    if landmarks is None:
        landmark_stats['fail'] += 1
        return torch.ones(512, GRID_SIZE, GRID_SIZE, dtype=torch.float32)
    landmark_stats['success'] += 1
    return _build_per_group(landmarks, img_np.shape[0], img_np.shape[1])

print('Landmark builders defined: face_alignment.')
print('Per-group attention: build_per_group_attention_fa.')

In [ ]:
# ── 512-Channel Cosine Similarity Relevance Scorer ───────────────────────────
# Core contribution: scores each of 512 channels by how much its 7×7
# activation map resembles the AU heatmap — no grid cell collapse.
#
# spatial_map : (N, 512, 7, 7) — ResNet Layer4 output
# au_heatmaps : (N, 7, 7)      — per-image AU heatmaps from landmarks
# Returns     : (N, 512)        — softmax-normalized relevance weights

def compute_channel_relevance_512(spatial_map, au_heatmaps):
    N, C, H, W = spatial_map.shape   # N, 512, 7, 7

    # Flatten spatial dims for cosine similarity: (N, 512, 49)
    feat_flat = spatial_map.view(N, C, -1)          # (N, 512, 49)

    # AU heatmap: (N, 7, 7) → (N, 1, 49) → broadcast to (N, 512, 49)
    au_flat   = au_heatmaps.view(N, 1, -1).expand(N, C, H*W)  # (N, 512, 49)

    # Cosine similarity per channel: dot / (norm_feat * norm_au)
    dot       = (feat_flat * au_flat).sum(dim=2)                      # (N, 512)
    norm_feat = feat_flat.norm(dim=2).clamp(min=1e-8)                 # (N, 512)
    norm_au   = au_flat.norm(dim=2).clamp(min=1e-8)                   # (N, 512)
    scores    = dot / (norm_feat * norm_au)                           # (N, 512)

    # Softmax: convert scores to attention weights summing to 1
    relevance_weights = F.softmax(scores, dim=1)                      # (N, 512)
    return relevance_weights

print('512-channel cosine similarity relevance scorer defined.')

In [ ]:
# ── AU-Weighted supervisor() ─────────────────────────────────────────────────
# l_sep weighted by AU cosine similarity scores | l_div unchanged

def supervisor_au_weighted(x, targets, relevance_weights, cnum=73):
    # ── l_div — unchanged from original CAFE ─────────────────────────
    branch   = x.reshape(x.size(0), x.size(1), 1, 1)
    branch   = my_MaxPool2d(kernel_size=(1, cnum), stride=(1, cnum))(branch)
    branch   = branch.reshape(x.size(0), -1, 1)
    loss_div = 1.0 - torch.mean(torch.sum(branch, dim=1)) / cnum

    # ── l_sep — reweighted by AU channel relevance ───────────────────
    branch_1 = x.reshape(x.size(0), x.size(1), 1, 1)
    branch_1 = branch_1 * relevance_weights.unsqueeze(-1).unsqueeze(-1)
    branch_1 = branch_1 * Mask(x.size(0))
    branch_1 = my_MaxPool2d(kernel_size=(1, cnum), stride=(1, cnum))(branch_1)
    branch_1 = branch_1.view(branch_1.size(0), -1)
    loss_sep = nn.CrossEntropyLoss()(branch_1, targets)

    return [loss_sep, loss_div]

print('supervisor_au_weighted defined.')


In [ ]:
# ── Batch Builders (face_alignment only) ────────────────────────────────────

def get_batch_au_heatmaps_fa(imgs_batch, labels_batch):
    """Single heatmap via face_alignment (GT label). Returns (N, 7, 7)."""
    heatmaps = []
    for i in range(imgs_batch.size(0)):
        img_np = denormalize_to_uint8(imgs_batch[i])
        expr_name = LABEL_TO_CLASS[labels_batch[i].item()]
        landmark_stats['total'] += 1
        expr = SYNONYM_MAP.get(expr_name, expr_name)
        if not AU_CODEBOOK.get(expr, []):
            landmark_stats['neutral_skip'] += 1
            heatmaps.append(torch.ones(GRID_SIZE, GRID_SIZE, dtype=torch.float32)); continue
        landmarks = get_landmarks_fa(img_np)
        if landmarks is None:
            landmark_stats['fail'] += 1
            heatmaps.append(torch.ones(GRID_SIZE, GRID_SIZE, dtype=torch.float32)); continue
        landmark_stats['success'] += 1
        heatmaps.append(build_heatmap_from_aus(landmarks, expr_name, img_np.shape[0], img_np.shape[1]))
    return torch.stack(heatmaps).to(imgs_batch.device)

get_batch_au_heatmaps = get_batch_au_heatmaps_fa

def get_batch_per_group_attention_fa(imgs_batch):
    """Per-group attention via face_alignment. Returns (N, 512, 7, 7)."""
    atts = []
    for i in range(imgs_batch.size(0)):
        atts.append(build_per_group_attention_fa(denormalize_to_uint8(imgs_batch[i])))
    return torch.stack(atts).to(imgs_batch.device)

print('Batch builders defined (face_alignment only).')


## 6. CAFE Model Architecture

In [ ]:
# ── Custom MaxPool2d (used in channel separation) ────────────────────
class my_MaxPool2d(Module):
    def __init__(self, kernel_size, stride=None, padding=0, dilation=1,
                 return_indices=False, ceil_mode=False):
        super(my_MaxPool2d, self).__init__()
        self.kernel_size    = kernel_size
        self.stride         = stride or kernel_size
        self.padding        = padding
        self.dilation       = dilation
        self.return_indices = return_indices
        self.ceil_mode      = ceil_mode

    def forward(self, input):
        input = input.transpose(3, 1)
        input = F.max_pool2d(input, self.kernel_size, self.stride,
                             self.padding, self.dilation, self.ceil_mode,
                             self.return_indices)
        input = input.transpose(3, 1).contiguous()
        return input


# ── Channel dropping mask ────────────────────────────────────────────
def Mask(nb_batch):
    """
    Generates random channel drop mask.
    7 expression groups x (63 or 64 channels each) = 512 total
    Drops 10 channels per group randomly.
    """
    bar = []
    for i in range(7):
        foo = [1] * 63 + [0] * 10
        if i == 6:
            foo = [1] * 64 + [0] * 10  # last group gets 1 extra channel
        random.shuffle(foo)
        bar += foo
    bar = [bar for _ in range(nb_batch)]
    bar = np.array(bar).astype('float32')
    bar = bar.reshape(nb_batch, 512, 1, 1)
    bar = torch.from_numpy(bar).to(device)
    bar = Variable(bar)
    return bar


# ── Channel separation + channel diverse loss ────────────────────────
def supervisor(x, targets, cnum=73):
    """
    Computes l_sep (channel separation loss) and l_div (channel diverse loss).
    x       : masked features (N, 512)
    targets : expression labels (N,)
    cnum    : channels per expression group (73 for groups 0-5, 74 for group 6)
    Returns: [l_sep, l_div]
    """
    # l_div — channel diverse loss
    branch = x
    branch = branch.reshape(branch.size(0), branch.size(1), 1, 1)
    branch = my_MaxPool2d(kernel_size=(1, cnum), stride=(1, cnum))(branch)
    branch = branch.reshape(branch.size(0), branch.size(1),
                            branch.size(2) * branch.size(3))
    loss_2 = 1.0 - 1.0 * torch.mean(torch.sum(branch, 2)) / cnum

    # l_sep — channel separation cross-entropy loss
    mask    = Mask(x.size(0))
    branch_1 = x.reshape(x.size(0), x.size(1), 1, 1) * mask
    branch_1 = my_MaxPool2d(kernel_size=(1, cnum), stride=(1, cnum))(branch_1)
    branch_1 = branch_1.view(branch_1.size(0), -1)
    loss_1   = nn.CrossEntropyLoss()(branch_1, targets)

    return [loss_1, loss_2]


# ── Custom ResNet-18 BasicBlock ──────────────────────────────────────
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)

        if downsample:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.downsample = None

    def forward(self, x):
        i = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        if self.downsample is not None:
            i = self.downsample(i)
        x += i
        return self.relu(x)


# ── Custom ResNet ────────────────────────────────────────────────────
class ResNet(nn.Module):
    def __init__(self, block, n_blocks, channels, output_dim):
        super().__init__()
        self.in_channels = channels[0]
        assert len(n_blocks) == len(channels) == 4

        self.conv1   = nn.Conv2d(3, self.in_channels, kernel_size=7,
                                  stride=2, padding=3, bias=False)
        self.bn1     = nn.BatchNorm2d(self.in_channels)
        self.relu    = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1  = self.get_resnet_layer(block, n_blocks[0], channels[0])
        self.layer2  = self.get_resnet_layer(block, n_blocks[1], channels[1], stride=2)
        self.layer3  = self.get_resnet_layer(block, n_blocks[2], channels[2], stride=2)
        self.layer4  = self.get_resnet_layer(block, n_blocks[3], channels[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(self.in_channels, output_dim)

    def get_resnet_layer(self, block, n_blocks, channels, stride=1):
        layers = []
        downsample = (self.in_channels != block.expansion * channels)
        layers.append(block(self.in_channels, channels, stride, downsample))
        for _ in range(1, n_blocks):
            layers.append(block(block.expansion * channels, channels))
        self.in_channels = block.expansion * channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        h = x.view(x.shape[0], -1)
        x = self.fc(h)
        return x, h

print('Utility classes defined (MaxPool2d, Mask, supervisor, BasicBlock, ResNet).')

In [ ]:
# ── CAFE Model — supports per-group spatial attention ─────────────────────────

class Model(nn.Module):
    def __init__(self, msceleb_path, num_classes=7, drop_rate=0):
        super(Model, self).__init__()
        res18 = ResNet(block=BasicBlock, n_blocks=[2, 2, 2, 2],
                       channels=[64, 128, 256, 512], output_dim=1000)
        msceleb_model = torch.load(msceleb_path, map_location='cpu', weights_only=False)
        res18.load_state_dict(msceleb_model['state_dict'], strict=False)
        print('MS-Celeb weights loaded.')
        self.drop_rate = drop_rate
        self.features  = nn.Sequential(*list(res18.children())[:-2])
        self.features2 = nn.Sequential(*list(res18.children())[-2:-1])
        fc_in_dim      = list(res18.children())[-1].in_features
        self.fc        = nn.Linear(fc_in_dim, num_classes)
        self.parm = {}
        for name, parameters in self.fc.named_parameters():
            self.parm[name] = parameters

    def forward(self, x, clip_model, targets, au_att=None, phase='train'):
        with torch.no_grad():
            image_features = clip_model.encode_image(x).float()     # (N, 512)

        spatial_map = self.features(x)                               # (N, 512, 7, 7)

        if au_att is not None:
            # au_att: (N, 512, 7, 7) per-group  OR  (N, 1, 7, 7) uniform
            x_pool = self.features2(spatial_map * au_att)
        else:
            x_pool = self.features2(spatial_map)

        x_flat          = x_pool.view(x_pool.size(0), -1)           # (N, 512)
        masked_features = image_features * torch.sigmoid(x_flat)    # (N, 512)
        out             = self.fc(masked_features)                   # (N, 7)

        if phase == 'train':
            return out, spatial_map, masked_features
        else:
            return out, out

print('Model defined (supports optional au_att for spatial attention).')

## Automated Cross-Dataset Training & Evaluation

Automatically trains on each dataset, tests on all datasets, and saves:
- Training curves (accuracy, loss)
- Per-test-dataset: accuracy, F1, classification report, confusion matrix
- AU heatmap & ResNet activation comparison
- AU alignment bar chart & detail table
- Channel relevance distribution & Jaccard overlap

In [ ]:
import kagglehub
import os

# ── Download all datasets ────────────────────────────────────────────
print('Downloading all datasets...')
RAW_PATHS = {
    'RAFDB':     kagglehub.dataset_download('shuvoalok/raf-db-dataset'),
    'FERPlus':   kagglehub.dataset_download('arnabkumarroy02/ferplus'),
    'AffectNET': kagglehub.dataset_download('mstjebashazida/affectnet'),
    'MMA':       kagglehub.dataset_download('mahmoudima/mma-facial-expression'),
    'SFEW':      kagglehub.dataset_download('vlntnstarodub/datasetsfew'),
}
print('All datasets downloaded.')

# ── Train configs (all 5 datasets) ───────────────────────────────────
TRAIN_CONFIGS = {
    'RAFDB': {
        'path': os.path.join(RAW_PATHS['RAFDB'], 'DATASET'),
        'train_map': raf_map,
        'train_phase': 'train',
        'val_phase': None,
    },
    'FERPlus': {
        'path': RAW_PATHS['FERPlus'],
        'train_map': ferplus_map,
        'train_phase': 'train',
        'val_phase': 'validation',
    },
    'AffectNET': {
        'path': os.path.join(RAW_PATHS['AffectNET'], 'archive (3)'),
        'train_map': affectnet_train_map,
        'train_phase': 'Train',
        'val_phase': None,
    },
    'MMA': {
        'path': os.path.join(RAW_PATHS['MMA'], 'MMAFEDB'),
        'train_map': mma_map,
        'train_phase': 'train',
        'val_phase': 'valid',
    },
    'SFEW': {
        'path': RAW_PATHS['SFEW'],
        'train_map': sfew_map,
        'train_phase': 'Train',
        'val_phase': None,
    },
}

# ── Test configs (all 5 datasets) ────────────────────────────────────
TEST_CONFIGS = {
    'RAFDB': {
        'path': os.path.join(RAW_PATHS['RAFDB'], 'DATASET'),
        'test_map': raf_map,
        'test_phase': 'test',
    },
    'FERPlus': {
        'path': RAW_PATHS['FERPlus'],
        'test_map': ferplus_map,
        'test_phase': 'test',
    },
    'AffectNET': {
        'path': os.path.join(RAW_PATHS['AffectNET'], 'archive (3)'),
        'test_map': affectnet_test_map,
        'test_phase': 'Test',
    },
    'MMA': {
        'path': os.path.join(RAW_PATHS['MMA'], 'MMAFEDB'),
        'test_map': mma_map,
        'test_phase': 'test',
    },
    'SFEW': {
        'path': RAW_PATHS['SFEW'],
        'test_map': sfew_map,
        'test_phase': 'Val',
    },
}

RESULTS_DIR = './results'
if os.path.exists('/content/drive'):
    RESULTS_DIR = '/content/drive/MyDrive/deeplearning/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Results will be saved to: {RESULTS_DIR}')
print(f'\nTrain datasets ({len(TRAIN_CONFIGS)}):')
for name in TRAIN_CONFIGS:
    print(f'  {name}')
print(f'Test datasets ({len(TEST_CONFIGS)}):')
for name in TEST_CONFIGS:
    print(f'  {name}')

In [ ]:
# ── FAST: Cached landmark batch builders (face_alignment only) ────────────────

def get_batch_au_heatmaps_fa_fast(idxs, labels_batch, cache, dataset):
    """AU heatmaps from cached FA landmarks. Returns (N, 7, 7)."""
    heatmaps = []
    for i in range(len(idxs)):
        idx = idxs[i].item() if hasattr(idxs[i], 'item') else idxs[i]
        expr_name = LABEL_TO_CLASS[labels_batch[i].item()]
        expr = SYNONYM_MAP.get(expr_name, expr_name)
        if not AU_CODEBOOK.get(expr, []):
            heatmaps.append(torch.ones(GRID_SIZE, GRID_SIZE, dtype=torch.float32)); continue
        lm = cache.get(idx, {}).get('fa', None)
        if lm is None:
            heatmaps.append(torch.ones(GRID_SIZE, GRID_SIZE, dtype=torch.float32)); continue
        heatmaps.append(build_heatmap_from_aus(lm, expr_name, 224, 224))
    return torch.stack(heatmaps).to(labels_batch.device)

def get_batch_per_group_attention_fa_fast(idxs, cache):
    """Per-group attention from cached FA landmarks. Returns (N, 512, 7, 7)."""
    atts = []
    for i in range(len(idxs)):
        idx = idxs[i].item() if hasattr(idxs[i], 'item') else idxs[i]
        lm = cache.get(idx, {}).get('fa', None)
        if lm is None:
            atts.append(torch.ones(512, GRID_SIZE, GRID_SIZE, dtype=torch.float32)); continue
        atts.append(_build_per_group(lm, 224, 224))
    return torch.stack(atts)

print('Fast cached batch builders defined (face_alignment only).')

# ── LIVE: re-detect landmarks from the (augmented) batch images ───────────────
# No cache. Runs face_alignment batch detection on the ACTUAL model inputs, so the
# attention map always aligns with whatever augmentation (flip/rotate/crop) was applied.
# Slower than the cached path, but geometry-consistent.

def get_batch_per_group_attention_fa_live(imgs_batch):
    """Per-group attention (N, 512, 7, 7) recomputed from augmented batch images."""
    np_imgs = [denormalize_to_uint8(imgs_batch[i]) for i in range(imgs_batch.size(0))]
    batch_tensor = torch.tensor(np.stack(np_imgs)).permute(0, 3, 1, 2).float().to(imgs_batch.device)
    try:
        batch_lm = fa.get_landmarks_from_batch(batch_tensor)
    except Exception:
        batch_lm = [None] * len(np_imgs)
    atts = []
    for i in range(len(np_imgs)):
        lm = batch_lm[i] if i < len(batch_lm) else None
        if lm is not None and isinstance(lm, list) and len(lm) > 0:
            fa_lm = np.array(lm[0] if isinstance(lm[0], np.ndarray) else lm)
        elif lm is not None and isinstance(lm, np.ndarray) and lm.shape == (68, 2):
            fa_lm = lm
        else:
            fa_lm = None
        if fa_lm is None:
            atts.append(torch.ones(512, GRID_SIZE, GRID_SIZE, dtype=torch.float32))
        else:
            atts.append(_build_per_group(fa_lm, 224, 224))
    return torch.stack(atts).to(imgs_batch.device)

print('Live (per-batch re-detection) attention builder defined.')


In [ ]:
# ── FAST: Train functions using cached landmarks (face_alignment only) ────────

def train_one_epoch_pergroup_fa_fast(model, train_loader, optimizer, scheduler, device, cache):
    model.train(); running_loss = 0.0; correct_sum = 0
    for imgs, labels, idxs, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        au_att = get_batch_per_group_attention_fa_fast(idxs, cache).to(device)
        output, spatial_map, masked_features = model(imgs, clip_model, labels, au_att=au_att, phase='train')
        mc_loss = supervisor(masked_features, labels, cnum=73)
        loss = nn.CrossEntropyLoss()(output, labels) + 1.5*mc_loss[0] + 5.0*mc_loss[1]
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        _, predicts = torch.max(output, 1)
        correct_sum += torch.eq(predicts, labels).sum(); running_loss += loss.item()
    scheduler.step()
    return (correct_sum.float()/len(train_loader.dataset)).item(), running_loss/len(train_loader)

print('Fast train function defined (face_alignment only).')


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

CLASS_NAMES_DISPLAY = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
NUM_CLASSES = 7
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 20

METHODS = {
    'BASELINE':           'baseline',
    'BASELINE_Aug':       'baseline_aug',
    'PerGroup_FA':        'pergroup_fa',
    'PerGroup_FA_Aug':    'pergroup_fa_aug',
}

# ── 1. BASELINE / BASELINE_Aug (no spatial attention, original supervisor) ─
#       (same train function; BASELINE_Aug only differs by Albumentations loader)
def train_one_epoch_baseline(model, train_loader, optimizer, scheduler, device):
    model.train(); running_loss = 0.0; correct_sum = 0
    for imgs, labels, _, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        output, spatial_map, masked_features = model(imgs, clip_model, labels, phase='train')
        mc_loss = supervisor(masked_features, labels, cnum=73)
        loss = nn.CrossEntropyLoss()(output, labels) + 1.5*mc_loss[0] + 5.0*mc_loss[1]
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        _, predicts = torch.max(output, 1)
        correct_sum += torch.eq(predicts, labels).sum(); running_loss += loss.item()
    scheduler.step()
    return (correct_sum.float()/len(train_loader.dataset)).item(), running_loss/len(train_loader)

# ── 2. PerGroup_FA (basic aug + LIVE landmark spatial attention, re-detected per batch) ─
def train_one_epoch_pergroup_fa(model, train_loader, optimizer, scheduler, device, cache):
    model.train(); running_loss = 0.0; correct_sum = 0
    for imgs, labels, idxs, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        # LIVE: re-detect landmarks from the augmented imgs (cache ignored — keeps attention aligned)
        au_att = get_batch_per_group_attention_fa_live(imgs).to(device)
        output, spatial_map, masked_features = model(imgs, clip_model, labels, au_att=au_att, phase='train')
        mc_loss = supervisor(masked_features, labels, cnum=73)
        loss = nn.CrossEntropyLoss()(output, labels) + 1.5*mc_loss[0] + 5.0*mc_loss[1]
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        _, predicts = torch.max(output, 1)
        correct_sum += torch.eq(predicts, labels).sum(); running_loss += loss.item()
    scheduler.step()
    return (correct_sum.float()/len(train_loader.dataset)).item(), running_loss/len(train_loader)

# ── 3. PerGroup_FA + Aug (albumentations + LIVE landmark spatial attention, re-detected per batch) ─
# Same as #2 but with albumentations DataLoader
train_one_epoch_pergroup_fa_aug = train_one_epoch_pergroup_fa  # same function, different loader

# ── Test ─────────────────────────────────────────────────────────────
def test_model(model, test_loader, device):
    model.eval()
    with torch.no_grad():
        running_loss = 0.0; iter_cnt = 0; correct_sum = 0; data_num = 0
        for imgs, labels, _, _ in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs, _ = model(imgs, clip_model, labels, phase='test')
            loss = nn.CrossEntropyLoss()(outputs, labels)
            iter_cnt += 1; _, predicts = torch.max(outputs, 1)
            correct_sum += torch.eq(predicts, labels).sum()
            running_loss += loss; data_num += outputs.size(0)
    return (correct_sum.float()/float(data_num)).item(), (running_loss/iter_cnt).item()

# ── Loaders ──────────────────────────────────────────────────────────
def create_loaders(cfg, is_train=True, use_albu=False):
    if is_train:
        t_transform = albu_transforms if use_albu else basic_transforms
        train_ds = FolderDataset(cfg['path'], phase=cfg['train_phase'],
                                transform=t_transform, label_map=cfg['train_map'], use_albu=use_albu)
        if cfg['val_phase'] is None:
            t_size = int(0.8*len(train_ds)); v_size = len(train_ds)-t_size
            g = torch.Generator().manual_seed(42)
            train_subset, _ = random_split(train_ds, [t_size, v_size], generator=g)
            eval_ds = FolderDataset(cfg['path'], phase=cfg['train_phase'],
                                   transform=eval_transforms, label_map=cfg['train_map'])
            eval_ds.phase = 'eval'
            g = torch.Generator().manual_seed(42)
            _, val_subset = random_split(eval_ds, [t_size, v_size], generator=g)
        else:
            train_subset = train_ds
            val_subset = FolderDataset(cfg['path'], phase=cfg['val_phase'],
                                      transform=eval_transforms, label_map=cfg['train_map'])
        return (data.DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
                data.DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS))
    else:
        test_ds = FolderDataset(cfg['path'], phase=cfg['test_phase'],
                               transform=eval_transforms, label_map=cfg['test_map'])
        return data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

def save_training_curves(history, save_dir, train_name, method_name):
    epochs_x = list(range(1, len(history['train_acc'])+1))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs_x, [a*100 for a in history['train_acc']], 'b-o', markersize=3, label='Train')
    axes[0].plot(epochs_x, [a*100 for a in history['val_acc']], 'r-o', markersize=3, label='Val')
    axes[0].set_title(f'Accuracy - {method_name} on {train_name}')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(epochs_x, history['train_loss'], 'b-o', markersize=3, label='Train')
    axes[1].plot(epochs_x, history['val_loss'], 'r-o', markersize=3, label='Val')
    axes[1].set_title(f'Loss - {method_name} on {train_name}')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout(); plt.savefig(os.path.join(save_dir, 'training_curves.png'), dpi=150); plt.close()

def evaluate_and_save(model, test_loader, save_dir, train_name, test_name, method_name):
    os.makedirs(save_dir, exist_ok=True); model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels, _, _ in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs, _ = model(imgs, clip_model, labels, phase='test')
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.cpu().numpy())
    acc = np.mean(np.array(all_preds)==np.array(all_labels))
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES_DISPLAY)
    with open(os.path.join(save_dir, 'results.txt'), 'w') as f:
        f.write(f'Train: {train_name} ({method_name}) -> Test: {test_name}\n')
        f.write(f'Accuracy: {acc*100:.2f}%\nF1 (macro): {f1_macro:.4f}\nF1 (weighted): {f1_weighted:.4f}\n\n')
        f.write(report)
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(all_labels, all_preds),
                           display_labels=CLASS_NAMES_DISPLAY).plot(cmap='Blues', xticks_rotation=90, ax=ax)
    ax.set_title(f'{method_name}: {train_name} -> {test_name}')
    plt.tight_layout(); plt.savefig(os.path.join(save_dir, 'confusion_matrix.png'), dpi=150); plt.close()

    # ── AU heatmap vs ResNet activation comparison (one sample per class) ──
    samples = {}
    with torch.no_grad():
        for imgs, labels, _, _ in test_loader:
            for i in range(imgs.size(0)):
                lbl = labels[i].item()
                if lbl not in samples: samples[lbl] = (imgs[i:i+1], labels[i:i+1])
            if len(samples) >= NUM_CLASSES: break
    avail = sorted(samples.keys()); n_cols = len(avail)
    if n_cols > 0:
        fig, axes = plt.subplots(3, n_cols, figsize=(3*n_cols, 9))
        if n_cols == 1: axes = axes.reshape(3, 1)
        cos_sims = []
        with torch.no_grad():
            for pi, ci in enumerate(avail):
                img, lbl = samples[ci]; img_d = img.to(device)
                # row 0: original face
                axes[0][pi].imshow(denormalize_to_uint8(img[0]))
                axes[0][pi].set_title(CLASS_NAMES_DISPLAY[ci]); axes[0][pi].axis('off')
                # row 1: GT-label AU heatmap (live landmarks)
                au_hm = get_batch_au_heatmaps_fa(img_d, lbl.to(device))
                axes[1][pi].imshow(au_hm[0].cpu().numpy(), cmap='hot', interpolation='bilinear'); axes[1][pi].axis('off')
                # row 2: ResNet spatial activation (channel-mean of layer4)
                sp = model.features(img_d)
                act = F.relu(sp[0].mean(dim=0)).cpu().numpy(); act = act / max(act.max(), 1e-6)
                axes[2][pi].imshow(act, cmap='hot', interpolation='bilinear'); axes[2][pi].axis('off')
                # cosine similarity between AU heatmap and ResNet activation
                cos = F.cosine_similarity(torch.tensor(act).flatten().unsqueeze(0),
                                          au_hm[0].cpu().flatten().unsqueeze(0)).item()
                cos_sims.append((CLASS_NAMES_DISPLAY[ci], cos))
                axes[2][pi].set_title(f'cos={cos:.3f}', fontsize=9, y=-0.18)
        for r, name in enumerate(['Original', 'AU Heatmap', 'ResNet Activation']):
            axes[r][0].set_ylabel(name, fontsize=11, rotation=90, labelpad=40, va='center')
            axes[r][0].axis('on'); axes[r][0].set_xticks([]); axes[r][0].set_yticks([])
        fig.suptitle(f'{method_name}: {train_name}->{test_name}', fontsize=13, y=1.02)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, 'au_vs_resnet.png'), dpi=150, bbox_inches='tight'); plt.close()
        with open(os.path.join(save_dir, 'au_cosine_similarity.txt'), 'w') as f:
            for name, cos in cos_sims: f.write(f'{name}: {cos:.3f}\n')
            if cos_sims: f.write(f'Average: {sum(c for _,c in cos_sims)/len(cos_sims):.3f}\n')

    return acc, f1_macro

print('All functions defined.')
print(f'Methods: {list(METHODS.keys())}')


In [ ]:
# ── ABLATION MAIN LOOP ────────────────────────────────────────────────
# BASELINE -> +Aug -> +PerGroup_FA -> +PerGroup_FA+Aug (incremental)

import gc

PATIENCE = 5
summary = []

def read_saved_results(results_path):
    acc, f1 = 0.0, 0.0
    with open(results_path, 'r') as f:
        for line in f:
            if line.startswith('Accuracy:'): acc = float(line.split(':')[1].strip().replace('%', '')) / 100
            elif line.startswith('F1 (macro):'): f1 = float(line.split(':')[1].strip())
    return acc, f1

for train_name, train_cfg in TRAIN_CONFIGS.items():
    print('\n' + '=' * 70)
    print(f'TRAIN DATASET: {train_name}')
    print('=' * 70)

    test_loaders = {}
    for test_name, test_cfg in TEST_CONFIGS.items():
        try: test_loaders[test_name] = create_loaders(test_cfg, is_train=False)
        except Exception as e: print(f'  SKIP test {test_name}: {e}')

    # cache no longer used for training (live re-detection); kept optional so Cell 23 can be skipped
    cache = globals().get('LANDMARK_CACHES', {}).get(f'{train_name}_train', {})
    train_base_dir = os.path.join(RESULTS_DIR, f'{train_name}_train')
    os.makedirs(train_base_dir, exist_ok=True)

    for method_name, method_type in METHODS.items():
        print(f'\n  --- {method_name} ---')
        method_dir = os.path.join(train_base_dir, method_name)
        os.makedirs(method_dir, exist_ok=True)
        model_path = os.path.join(method_dir, 'best_model.pth')

        # Method-specific DataLoader
        use_albu = method_type in ('pergroup_fa_aug', 'baseline_aug')
        try:
            train_loader, val_loader = create_loaders(train_cfg, is_train=True, use_albu=use_albu)
        except Exception as e:
            print(f'    SKIP: {e}'); continue

        if os.path.exists(model_path):
            print(f'    [SKIP TRAINING] best_model.pth found, loading...')
            setup_seed(3407)
            model = Model(msceleb_path=MSCELEB_PATH, num_classes=NUM_CLASSES)
            model.to(device)
            ckpt = torch.load(model_path, map_location=device, weights_only=False)
            model.load_state_dict(ckpt['model_state_dict'])
        else:
            setup_seed(3407)
            model = Model(msceleb_path=MSCELEB_PATH, num_classes=NUM_CLASSES)
            model.to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.0002, weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

            history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}
            best_acc = 0.0; patience_counter = 0

            for epoch in range(1, EPOCHS + 1):
                if method_type in ('baseline', 'baseline_aug'):
                    train_acc, train_loss = train_one_epoch_baseline(model, train_loader, optimizer, scheduler, device)
                else:  # pergroup_fa or pergroup_fa_aug (same train function, different loader)
                    train_acc, train_loss = train_one_epoch_pergroup_fa(model, train_loader, optimizer, scheduler, device, cache)

                val_acc, val_loss = test_model(model, val_loader, device)
                history['train_acc'].append(train_acc); history['train_loss'].append(train_loss)
                history['val_acc'].append(val_acc); history['val_loss'].append(val_loss)
                aug_tag = ' [albu]' if use_albu else ''
                print(f'    Epoch [{epoch:>2}/{EPOCHS}] Train: {train_acc*100:5.2f}% | Val: {val_acc*100:5.2f}%{aug_tag}')

                if val_acc > best_acc:
                    best_acc = val_acc; patience_counter = 0
                    torch.save({'model_state_dict': model.state_dict()}, model_path)
                else:
                    patience_counter += 1
                    if patience_counter >= PATIENCE:
                        print(f'    Early stopping at epoch {epoch}'); break

            save_training_curves(history, method_dir, train_name, method_name)
            print(f'    Best val acc: {best_acc*100:.2f}%')
            ckpt = torch.load(model_path, map_location=device, weights_only=False)
            model.load_state_dict(ckpt['model_state_dict'])

        for test_name, test_loader in test_loaders.items():
            test_dir = os.path.join(method_dir, f'{test_name}_test')
            results_file = os.path.join(test_dir, 'results.txt')
            if os.path.exists(results_file):
                acc, f1 = read_saved_results(results_file)
                summary.append((train_name, method_name, test_name, acc, f1))
                print(f'    [SKIP TEST] {test_name}: Acc={acc*100:.2f}% F1={f1:.4f}')
            else:
                print(f'    Testing on: {test_name}')
                acc, f1 = evaluate_and_save(model, test_loader, test_dir, train_name, test_name, method_name)
                summary.append((train_name, method_name, test_name, acc, f1))
                print(f'      Acc={acc*100:.2f}% F1={f1:.4f}')

        # ── free GPU memory before next model (prevents cross-run OOM accumulation) ──
        del model
        if 'optimizer' in globals(): del optimizer
        if 'scheduler' in globals(): del scheduler
        gc.collect(); torch.cuda.empty_cache()

print('\n' + '=' * 70)
print('ABLATION STUDY COMPLETE')
print('=' * 70)


In [ ]:
# ── Ablation Results with Average ─────────────────────────────────────
import pandas as pd

test_names_ordered = ['RAFDB', 'FERPlus', 'AffectNET', 'MMA', 'SFEW']
method_names = list(METHODS.keys())

lookup = {}
for train_n, method_n, test_n, acc, f1 in summary:
    lookup[(train_n, method_n, test_n)] = (acc * 100, f1)

for train_name in TRAIN_CONFIGS.keys():
    print(f'\n{"="*100}')
    print(f'  Train: {train_name}')
    print(f'{"="*100}')
    header = f'{"Method":<22}'
    for tn in test_names_ordered: header += f'| {tn:^14} '
    header += f'| {"Mean":^14} '
    print(header)
    sub = f'{"":<22}'
    for _ in test_names_ordered: sub += f'| {"acc":>5} {"F1":>6} '
    sub += f'| {"acc":>5} {"F1":>6} '
    print(sub); print('-' * len(header))

    for method_name in method_names:
        row = f'{method_name:<22}'
        accs = []; f1s = []
        for tn in test_names_ordered:
            key = (train_name, method_name, tn)
            if key in lookup:
                acc, f1 = lookup[key]
                row += f'| {acc:>5.1f} {f1:>6.2f} '
                accs.append(acc); f1s.append(f1)
            else:
                row += f'| {"N/A":>5} {"N/A":>6} '
        # Average
        if accs:
            row += f'| {np.mean(accs):>5.1f} {np.mean(f1s):>6.2f} '
        else:
            row += f'| {"N/A":>5} {"N/A":>6} '
        print(row)

# Save CSV with average
rows_csv = []
for train_n, method_n, test_n, acc, f1 in summary:
    rows_csv.append({'train': train_n, 'method': method_n, 'test': test_n,
                     'acc': round(acc*100, 2), 'macro_f1': round(f1, 2)})
df = pd.DataFrame(rows_csv)

# Add average per train/method
avg_rows = []
for train_n in TRAIN_CONFIGS.keys():
    for method_n in method_names:
        sub = df[(df['train']==train_n) & (df['method']==method_n)]
        if len(sub) > 0:
            avg_rows.append({'train': train_n, 'method': method_n, 'test': 'Mean',
                            'acc': round(sub['acc'].mean(), 2), 'macro_f1': round(sub['macro_f1'].mean(), 2)})
df_full = pd.concat([df, pd.DataFrame(avg_rows)], ignore_index=True)
df_full.to_csv(os.path.join(RESULTS_DIR, 'ablation_results.csv'), index=False)

# Pivot with average
for metric, col in [('Accuracy (%)', 'acc'), ('Macro F1', 'macro_f1')]:
    pivot = df_full.pivot_table(index=['train', 'method'], columns='test', values=col)
    cols_order = [t for t in test_names_ordered + ['Mean'] if t in pivot.columns]
    pivot = pivot.reindex(columns=cols_order)
    print(f'\n{metric}:')
    print(pivot.to_string(float_format='%.2f'))

# Heatmap per train dataset
for train_name in TRAIN_CONFIGS.keys():
    sub = df[df['train'] == train_name]
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    for ax_idx, (metric, col, vmax, fmt) in enumerate([
        ('Accuracy (%)', 'acc', 100, '.1f'), ('Macro F1', 'macro_f1', 1.0, '.2f')
    ]):
        pivot = sub.pivot(index='method', columns='test', values=col)
        # Add mean column
        pivot['Mean'] = pivot.mean(axis=1)
        cols = [t for t in test_names_ordered if t in pivot.columns] + ['Mean']
        pivot = pivot.reindex(index=method_names, columns=cols)
        ax = axes[ax_idx]
        im = ax.imshow(pivot.values.astype(float), cmap='YlGn', vmin=0, vmax=vmax, aspect='auto')
        ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=45)
        ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
        ax.set_title(f'{metric} (Train: {train_name})')
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                v = pivot.iloc[i, j]
                ax.text(j, i, f'{v:{fmt}}', ha='center', va='center', fontsize=10,
                        color='white' if v > vmax*0.5 else 'black')
        plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'{train_name}_ablation.png'), dpi=150, bbox_inches='tight')
    plt.show()

print(f'\nAll results saved to: {RESULTS_DIR}')


In [ ]:
# ── Ablation Analysis Visualizations ──────────────────────────────────────────

# 1. Incremental Gain Bar Chart (per train dataset)
for train_name in TRAIN_CONFIGS.keys():
    accs = {}
    for method_name in method_names:
        vals = [lookup.get((train_name, method_name, tn), (0,0))[0] for tn in test_names_ordered]
        accs[method_name] = vals

    # Cross-dataset mean (exclude in-dataset)
    in_idx = test_names_ordered.index(train_name) if train_name in test_names_ordered else -1
    cross_means = {}
    for mn in method_names:
        cross = [accs[mn][i] for i in range(len(test_names_ordered)) if i != in_idx]
        cross_means[mn] = np.mean(cross) if cross else np.mean(accs[mn])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # a) Per-dataset accuracy comparison
    x = np.arange(len(test_names_ordered))
    n_m = len(method_names)
    w = 0.8 / n_m
    colors = ['#4472C4', '#FFC000', '#ED7D31', '#70AD47']
    for mi, mn in enumerate(method_names):
        axes[0].bar(x + mi*w, accs[mn], w, label=mn, color=colors[mi % len(colors)], alpha=0.85)
    axes[0].set_xticks(x + w*(n_m-1)/2); axes[0].set_xticklabels(test_names_ordered)
    axes[0].set_ylabel('Accuracy (%)'); axes[0].set_title(f'Per-Dataset Accuracy (Train: {train_name})')
    axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)

    # b) Incremental gain (cross-dataset mean)
    base_val = cross_means[method_names[0]]
    gains = [0]
    for mn in method_names[1:]:
        gains.append(cross_means[mn] - base_val)
    bar_colors = [colors[i % len(colors)] for i in range(len(method_names))]
    bars = axes[1].bar(method_names, [cross_means[mn] for mn in method_names], color=bar_colors, alpha=0.85)
    for bar, mn in zip(bars, method_names):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                    f'{cross_means[mn]:.1f}', ha='center', fontsize=10)
    # Add delta annotations
    for i in range(1, len(method_names)):
        delta = cross_means[method_names[i]] - cross_means[method_names[i-1]]
        sign = '+' if delta >= 0 else ''
        axes[1].annotate(f'{sign}{delta:.1f}%', xy=(i, cross_means[method_names[i]]),
                        xytext=(i-0.3, cross_means[method_names[i]]+2),
                        fontsize=9, color='red', fontweight='bold',
                        arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
    axes[1].set_ylabel('Cross-Dataset Mean Acc (%)')
    axes[1].set_title(f'Incremental Contribution (Train: {train_name})')
    axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'{train_name}_incremental_gain.png'), dpi=150, bbox_inches='tight')
    plt.show()

# 2. Cross-Dataset Generalization Gap
print('\n=== Cross-Dataset Generalization Gap ===')
print(f'{"Train":<12} {"Method":<22} {"In-dataset":>10} {"Cross-mean":>12} {"Gap":>8}')
print('-' * 66)
for train_name in TRAIN_CONFIGS.keys():
    for method_name in method_names:
        in_key = (train_name, method_name, train_name)
        in_acc = lookup.get(in_key, (0,0))[0]
        cross_accs = [lookup.get((train_name, method_name, tn), (0,0))[0]
                     for tn in test_names_ordered if tn != train_name]
        cross_mean = np.mean(cross_accs) if cross_accs else 0
        gap = in_acc - cross_mean
        print(f'{train_name:<12} {method_name:<22} {in_acc:>10.1f} {cross_mean:>12.1f} {gap:>8.1f}')

# 3. Generalization Gap Chart
fig, ax = plt.subplots(figsize=(12, 6))
train_names_list = list(TRAIN_CONFIGS.keys())
x = np.arange(len(train_names_list))
n_m = len(method_names)
w = 0.8 / n_m
colors = ['#4472C4', '#FFC000', '#ED7D31', '#70AD47']

for mi, mn in enumerate(method_names):
    gaps = []
    for tn in train_names_list:
        in_acc = lookup.get((tn, mn, tn), (0,0))[0]
        cross = [lookup.get((tn, mn, t), (0,0))[0] for t in test_names_ordered if t != tn]
        gaps.append(in_acc - np.mean(cross) if cross else 0)
    ax.bar(x + mi*w, gaps, w, label=mn, color=colors[mi % len(colors)], alpha=0.85)

ax.set_xticks(x + w*(n_m-1)/2); ax.set_xticklabels(train_names_list)
ax.set_ylabel('Gap (In-dataset - Cross-dataset mean)')
ax.set_title('Generalization Gap (lower = better generalization)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'generalization_gap.png'), dpi=150, bbox_inches='tight')
plt.show()

# 4. Per-Class Accuracy Improvement (RAFDB example)
if 'RAFDB' in TRAIN_CONFIGS:
    print('\n=== Per-Class Accuracy Change (Train: RAFDB) ===')
    for test_name in test_names_ordered:
        baseline_dir = os.path.join(RESULTS_DIR, f'RAFDB_train/BASELINE/{test_name}_test/results.txt')
        best_dir = os.path.join(RESULTS_DIR, f'RAFDB_train/{method_names[-1]}/{test_name}_test/results.txt')
        if os.path.exists(baseline_dir) and os.path.exists(best_dir):
            b_acc, b_f1 = read_saved_results(baseline_dir)
            p_acc, p_f1 = read_saved_results(best_dir)
            delta_acc = (p_acc - b_acc) * 100
            delta_f1 = p_f1 - b_f1
            sign_a = '+' if delta_acc >= 0 else ''
            sign_f = '+' if delta_f1 >= 0 else ''
            print(f'  {test_name:>10}: Acc {sign_a}{delta_acc:.1f}% | F1 {sign_f}{delta_f1:.4f}')

print('\nAblation analysis complete.')

In [ ]:
import shutil

# 'results' 폴더 전체를 'results_download.zip'으로 압축
shutil.make_archive('results_download', 'zip', 'results')